# Solution 04 — the temperature agent

Two exact conversion tools, one agent that must use them, and proof from the messages that they ran.

In [ ]:
from learningdantic import setup

MODEL = setup()

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.messages import ToolReturnPart

temp_agent = Agent(
    MODEL,
    instructions=(
        "You compare temperatures. You never convert units yourself: always call a conversion tool "
        "first, then compare the numbers it returns."
    ),
)


@temp_agent.tool_plain
def c_to_f(celsius: float) -> float:
    """Convert a temperature from Celsius to Fahrenheit, exactly."""
    return celsius * 9 / 5 + 32


@temp_agent.tool_plain
def f_to_c(fahrenheit: float) -> float:
    """Convert a temperature from Fahrenheit to Celsius, exactly."""
    return (fahrenheit - 32) * 5 / 9


result = await temp_agent.run("Is 25°C warmer than 80°F? Show both on one scale.")
print(result.output)

tools_used = [
    part.tool_name
    for message in result.all_messages()
    for part in message.parts
    if isinstance(part, ToolReturnPart)
]
print("tools that actually ran:", tools_used)

In [ ]:
assert c_to_f(25) == 77
assert round(f_to_c(80), 2) == 26.67
assert {"c_to_f", "f_to_c"} & set(tools_used), "the agent must convert with a tool"
assert any(value in result.output for value in ("77", "26.6", "26.7")), result.output

**Notes**

- 25°C is 77°F, so 25°C is *cooler* than 80°F. Models often get this kind of arithmetic wrong, which is why
  the conversion lives in your tools.
- If your agent answered without tools, its `instructions` weren't firm enough. Tightening them is normal
  prompt work, not a framework bug.